# 05. LoRA и QLoRA

Три отдельных запуска с r=8,16,32. По умолчанию здесь QLoRA для Colab T4; без --quantized получится обычная LoRA. Обучаемся на train, checkpoint выбираем по val loss, ранги сравниваем по val F1.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Open the notebook from the CodeGuard repository"

def run_script(name, *args):
    subprocess.run([sys.executable, str(ROOT / "src" / name), *map(str, args)], cwd=ROOT, check=True)


In [ ]:
for rank in (8, 16, 32):
    run_script("train_lora.py", "--quantized", "--rank", rank, "--output", f"runs/qlora_r{rank}")

In [ ]:
import json
import pandas as pd
rows = []
for rank in (8, 16, 32):
    metrics = json.loads((ROOT / f"runs/qlora_r{rank}/metrics.json").read_text(encoding="utf-8"))
    rows.append({key: metrics[key] for key in ("rank", "accuracy", "precision", "recall", "f1", "roc_auc")})
comparison = pd.DataFrame(rows).sort_values("f1", ascending=False)
comparison

После выбора ранга зафиксируйте настройки до итоговой оценки test. Каждый каталог содержит adapter, checkpoints, metrics.json и predictions.csv. Для повторной оценки адаптера используйте llm_baseline.py --adapter runs/qlora_r16/adapter с теми же model/revision/max-length/quantized. Не выбирайте ранг по test.